In [11]:
import pandas as pd
import requests

url = "https://data.cdc.gov/resource/i46a-9kgh.json"
params = {
    "$select": "stateabbr,countyname,countyfips,cholscreen_adjprev",
    "stateabbr": "CA"
}

r = requests.get(url, params=params)
r.raise_for_status()

chol_ca = pd.DataFrame(r.json())

chol_ca

,stateabbr,countyname,countyfips,cholscreen_adjprev
0,CA,Solano,06095,84.9
1,CA,Placer,06061,86.0
2,CA,Kings,06031,81.1
3,CA,Fresno,06019,82.3
4,CA,Amador,06005,84.0
5,CA,Santa Barbara,06083,83.9
6,CA,Orange,06059,86.0
7,CA,Modoc,06049,82.3
8,CA,Tehama,06103,82.1
9,CA,San Benito,06069,84.1


In [13]:
# read income data
income_data = pd.read_csv("26-the-backpropagators-analysis/cholestrol_income/data/raw/income20261533.csv", skiprows=4)

income_data

,Area,Year,Period,Income Type,Income,Population
0,California,2023,Annual,Total Personal Income - BEA,"$3,166,135,354",39123861
1,California,2023,Annual,Per Capita Personal Income - BEA,"$81,255",39123861
2,Alameda County,2023,Annual,Total Personal Income - BEA,"$173,018,284",1644199
3,Alameda County,2023,Annual,Per Capita Personal Income - BEA,"$106,657",1644199
4,Alpine County,2023,Annual,Total Personal Income - BEA,"$86,026",1166
...,...,...,...,...,...,...
113,Ventura County,2023,Annual,Per Capita Personal Income - BEA,"$78,091",826745
114,Yolo County,2023,Annual,Total Personal Income - BEA,"$14,947,986",221202
115,Yolo County,2023,Annual,Per Capita Personal Income - BEA,"$67,778",221202
116,Yuba County,2023,Annual,Total Personal Income - BEA,"$4,336,377",83405


# Merge datasets

In [15]:
# Keep one income row per county before merging.
income_per_capita = income_data.loc[
    income_data["Income Type"].eq("Per Capita Personal Income - BEA")
].copy()

income_per_capita = income_per_capita.loc[
    income_per_capita["Area"].ne("California")
].copy()

income_per_capita["countyname"] = income_per_capita["Area"].str.replace(
    r"\s+County$", "", regex=True
).str.strip()
income_per_capita["per_capita_income"] = (
    income_per_capita["Income"]
    .replace({r"[$,]": ""}, regex=True)
    .astype(int)
)
income_per_capita["population"] = pd.to_numeric(
    income_per_capita["Population"], errors="coerce"
).astype("Int64")

income_per_capita = income_per_capita.rename(columns={"Year": "income_year"})[
    ["countyname", "income_year", "per_capita_income", "population"]
]

chol_clean = chol_ca.copy()
chol_clean["countyname"] = chol_clean["countyname"].str.strip()
chol_clean["countyfips"] = chol_clean["countyfips"].astype(str).str.zfill(5)
chol_clean["cholscreen_adjprev"] = pd.to_numeric(
    chol_clean["cholscreen_adjprev"], errors="coerce"
)

missing_from_income = sorted(
    set(chol_clean["countyname"]) - set(income_per_capita["countyname"])
)
missing_from_cholesterol = sorted(
    set(income_per_capita["countyname"]) - set(chol_clean["countyname"])
)

if missing_from_income or missing_from_cholesterol:
    raise ValueError(
        f"County mismatch. Missing income: {missing_from_income}; "
        f"missing cholesterol: {missing_from_cholesterol}"
    )

merged_data = (
    chol_clean.merge(
        income_per_capita,
        on="countyname",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("countyname")
    .reset_index(drop=True)
)

merged_data.to_csv("26-the-backpropagators-analysis/cholestrol_income/data/processed/cholesterol_income_by_county.csv", index=False)

merged_data

,stateabbr,countyname,countyfips,cholscreen_adjprev,income_year,per_capita_income,population
0,CA,Alameda,06001,86.5,2023,106657,1644199
1,CA,Alpine,06003,85.2,2023,75395,1166
2,CA,Amador,06005,84.0,2023,50020,40028
3,CA,Butte,06007,83.3,2023,56847,205741
4,CA,Calaveras,06009,83.6,2023,58425,44616
5,CA,Colusa,06011,81.8,2023,58303,21981
6,CA,Contra Costa,06013,85.8,2023,103218,1148745
7,CA,Del Norte,06015,82.3,2023,47141,26320
8,CA,El Dorado,06017,85.4,2023,84533,189355
9,CA,Fresno,06019,82.3,2023,52728,1017152
